# Hermes Drive → ChatGPT bridge

Authority-less transport worker. Reads existing Google Drive archive parts and creates verified chunks below ChatGPT's 256 MiB connector ceiling. Originals are never deleted, renamed, or modified.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import hashlib, json, os, re, time

ROOT = Path('/content/drive/MyDrive')
OUTROOT = ROOT / 'HERMES_CHATGPT_BRIDGE'
OUTROOT.mkdir(parents=True, exist_ok=True)

CHUNK_BYTES = 200_000_000  # safely below 256 MiB connector ceiling
families = (
    'hermes_ultra_20251126-212657.tar.zst.part.',
    'staging_ultra_20251126-222340.tar.zst.part.',
)

sources = sorted(
    p for p in ROOT.iterdir()
    if p.is_file() and p.name.startswith(families) and re.search(r'\.part\.\d{3}$', p.name)
)

print('[Hermes][SOURCES]')
for p in sources:
    print(p.name, p.stat().st_size)

if not sources:
    raise SystemExit('No matching Hermes ultra archive parts found in My Drive root')

def sha256_file(path, block=4*1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(block), b''):
            h.update(b)
    return h.hexdigest()

batch = {
    'schema': 'hermes.chatgpt.drive_chunk_bridge.v1',
    'created_unix': int(time.time()),
    'chunk_bytes': CHUNK_BYTES,
    'sources': [],
    'policy': {
        'source_delete': 'NO', 'source_rename': 'NO', 'source_modify': 'NO',
        'repo_authority': 'NO', 'canon': 'NO'
    }
}

for src in sources:
    st = src.stat()
    outdir = OUTROOT / (src.name + '__chunks_200000000')
    outdir.mkdir(parents=True, exist_ok=True)
    manifest = outdir / 'manifest.tsv'
    done = outdir / 'DONE.json'

    if done.exists():
        try:
            old = json.loads(done.read_text())
            if old.get('source_bytes') == st.st_size:
                print('[Hermes][SKIP_VERIFIED_EXISTING]', src.name)
                batch['sources'].append(old)
                continue
        except Exception:
            pass

    whole = hashlib.sha256()
    rows = []
    offset = 0
    idx = 0

    with open(src, 'rb') as fin:
        while True:
            data = fin.read(CHUNK_BYTES)
            if not data:
                break

            whole.update(data)
            ch = hashlib.sha256(data).hexdigest()
            name = f'{src.name}.subpart.{idx:03d}'
            final = outdir / name
            tmp = outdir / (name + '.partial')

            with open(tmp, 'wb') as fout:
                fout.write(data)
                fout.flush()
                os.fsync(fout.fileno())
            os.replace(tmp, final)

            rb = sha256_file(final)
            if rb != ch or final.stat().st_size != len(data):
                raise RuntimeError(f'readback mismatch: {final}')

            rows.append({
                'index': idx,
                'offset': offset,
                'bytes': len(data),
                'sha256': ch,
                'readback_sha256': rb,
                'name': name,
            })
            print('[Hermes][CHUNK]', src.name, idx, offset, len(data), ch)
            offset += len(data)
            idx += 1

    source_sha = whole.hexdigest()
    if offset != st.st_size:
        raise RuntimeError(f'source byte accounting mismatch: {src}')

    with open(manifest, 'w', encoding='utf-8') as mf:
        mf.write('index\toffset\tbytes\tsha256\treadback_sha256\tname\n')
        for r in rows:
            mf.write(f"{r['index']}\t{r['offset']}\t{r['bytes']}\t{r['sha256']}\t{r['readback_sha256']}\t{r['name']}\n")

    rec = {
        'source_name': src.name,
        'source_bytes': st.st_size,
        'source_mtime_ns': st.st_mtime_ns,
        'source_sha256': source_sha,
        'chunk_bytes': CHUNK_BYTES,
        'chunk_count': len(rows),
        'chunk_total_bytes': sum(r['bytes'] for r in rows),
        'output_dir': outdir.name,
        'manifest': manifest.name,
        'readback': 'PASS',
        'source_delete': 'NO',
        'source_modify': 'NO',
    }
    done.write_text(json.dumps(rec, indent=2) + '\n', encoding='utf-8')
    batch['sources'].append(rec)
    print('[Hermes][SOURCE_DONE]', json.dumps(rec, indent=2))

ready = OUTROOT / 'READY_FOR_CHATGPT.json'
ready.write_text(json.dumps(batch, indent=2) + '\n', encoding='utf-8')
print('\n[Hermes][DONE]')
print('output=', OUTROOT)
print('ready=', ready)
print('sources=', len(batch['sources']))
